<a href="https://colab.research.google.com/github/kyungjunoh1/LLM-workspace/blob/main/2_Fine_tuning_%EB%B0%8F_pipline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 미세 조정(fine-tuning)
- 이미 학습된 큰 모델을 특정 목적이나 데이터에 맞게 조금 더 학습시키는 것
- 큰 모델은 일반적인 언어 이해 능력을 갖고 있다. 하지만 어떤 특정 작업에는 바로 쓰기 어렵다
  - 예) 의료 문서를 분류, 특정 회사 이메일 자동 작성 등
- 그래서 이미 배운 일반 지식을 유지하면서, 특정 데이터에 맞게 "살짝 조정"하는 것
---
### 미세 조정을 하는 이유
- 시간과 자원을 절약 : 처음부터 모델을 학습시키는 것보다 훨씬 빠르고 효율적
- 성능 향상 : 특정 업무나 도메인에 맞춰 모델 성능을 높일 수 있다
- 적은 데이터로 가능 : 일반적인 큰 모델이 이미 언어를 잘 알기 때문에, 상대적으로 작은 데이터셋으로도 충분히 학습 가능
---
### 장 / 단점
#### 장점
- 빠른 처리(이미 학습된 모델이라 금방 적용 됨)
- 일관성 있음( 항상 같은 스타일로 답변)
- 특정 분야에 강함(ERP, 의료, 법률)
#### 단점
- 비용이 큼(이미 학습된 대규모 모델에 추가 데이터 처리하기 때문에 시간이 오래 걸림)
- 수정이 어려움( 최신 데이터 변경시 다시 학습해야 한다)

In [ ]:
!pip install transformers==4.56.1 -qqq
!pip install huggingface_hub==0.34.4 -qqq
!pip install datasets==4.0.0 -qqq #Dataset 자료형 사용하기 위한 설정

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 13.6 MB/s eta 0:00:00


In [ ]:
import transformers
import huggingface_hub
import datasets

#import warnings
#warnings.filterwarnings('ignore')

print(f"transformers : {transformers.__version__}" )
print(f"huggingface_hub : {huggingface_hub.__version__}" )
print(f"datasets : {datasets.__version__}" )

transformers : 4.56.1
huggingface_hub : 0.34.4
datasets : 4.0.0


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
model_id ="kakaocorp/kanana-nano-2.1b-base"
tokenizer = AutoTokenizer.from_pretrained( model_id )
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto" )

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
import torch
torch.cuda.is_available()
next( model.parameters() ).device

device(type='cuda', index=0)

In [ ]:
prompt = "삼성전자 설명"
inputs = tokenizer( prompt, return_tensors="pt" ).to('cuda')

In [ ]:
for k, v in inputs.items():
  print(k,":", v.device)

input_ids : cuda:0
attention_mask : cuda:0


In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True, # 추론 결과를 다양하게 생성
    top_p=0.9, # 상위 노출되는 값
    temperature=0.4, # do_sample이 True로 설정 되었을 때 나온 내용들을 자유롭게 생성, 값이 크면 상상력 발휘
    pad_token_id=tokenizer.eos_token_id
)

In [ ]:
answer = tokenizer.batch_decode(outputs[0], skip_special_tokens=True)

In [ ]:
answer

['삼성전자 설명회\n삼성전자 설명회에 다녀왔습니다.  삼성전자 설명회는 삼성전자 본사에서 진행되는데요. 삼성전자 본사는 서울 서초구에 위치해 있습니다.  삼성전자 본사에 도착하니, 삼성전자 직원들이 반갑게 맞아주었습니다.  삼성전자 본사 건물은 삼성전자뿐만 아니라 삼성전자 계열사들이 함께 사용하는 건물입니다.  삼성전자 본사 건물']

In [ ]:
# 미세조정할 학습 데이터 생성
train_data = [
    {"text": "발주 수량을 어떻게 추천해?\n최근 판매량, 요일 패턴, 프로모션 여부, 현재 재고를 함께 반영해서 추천합니다."},
    {"text": "편의점 재고 부족 원인이 뭐야?\n최근 4주 판매량 증가와 발주 리드타임 지연이 동시에 발생했기 때문입니다."},
]

In [ ]:
type(train_data)

list

### Dataset 자료형
- 데이터 학습하고자 하는 경우 Dataset 자료형 사용
### 학습에 필요한 데이터
- 문자가 아닌 숫자로 변환된 데이터
- input_ids, attention_mask, labels
  - input_ids : 사용자 입력한 값
  - labels 정답. labels값과 llm이 예측한 값과 비교하기 위한 값. 정답에 가깝게 llm이 추론

In [ ]:
train_data

[{'text': '발주 수량을 어떻게 추천해?\n최근 판매량, 요일 패턴, 프로모션 여부, 현재 재고를 함께 반영해서 추천합니다.'},
 {'text': '편의점 재고 부족 원인이 뭐야?\n최근 4주 판매량 증가와 발주 리드타임 지연이 동시에 발생했기 때문입니다.'}]

In [ ]:
token = tokenizer(train_data[0]["text"])
token['labels'] = token["input_ids"].copy()
token

{'input_ids': [128000, 102133, 55430, 29833, 104690, 18359, 112655, 109336, 34983, 5380, 104156, 104152, 116604, 104690, 11, 87097, 33177, 108158, 95252, 11, 108360, 101555, 93131, 84618, 64189, 11, 111530, 102888, 35495, 18918, 106999, 64857, 101090, 97237, 109336, 61938, 13], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [128000, 102133, 55430, 29833, 104690, 18359, 112655, 109336, 34983, 5380, 104156, 104152, 116604, 104690, 11, 87097, 33177, 108158, 95252, 11, 108360, 101555, 93131, 84618, 64189, 11, 111530, 102888, 35495, 18918, 106999, 64857, 101090, 97237, 109336, 61938, 13]}

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_function(examples):
  tokens = tokenizer(examples["text"], padding=True)
  tokens['labels'] = tokens["input_ids"].copy()
  return tokens

In [ ]:
from datasets import Dataset
dataset = Dataset.from_list(train_data)
dataset

Dataset({
    features: ['text'],
    num_rows: 2
})

In [ ]:
dataset = dataset.map(tokenize_function)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [ ]:
dataset

Dataset({
    features: ['text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2
})

In [ ]:
dataset = Dataset.from_list(train_data)
dataset

Dataset({
    features: ['text'],
    num_rows: 2
})

In [ ]:
def tokenize_function(examples):
  tokens = tokenizer(examples["text"]) #padding=True)
 #tokens['labels'] = tokens["input_ids"].copy()
  return tokens

In [ ]:
dataset = dataset.map(tokenize_function)
dataset

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 2
})

In [ ]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
    )

In [ ]:
from transformers import Trainer, TrainingArguments

In [ ]:
args = TrainingArguments(
    report_to = "none"
)

In [ ]:
trainer = Trainer(
    model = model,
    train_dataset = dataset,
    args = args,
    data_collator = data_collator
)

In [ ]:
trainer.train()

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 1.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 14.14 GiB is allocated by PyTorch, and 286.65 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### OOM(Out of Memory)
- train 진행시 메모리 부족현상 발생
- 한정된 GPU 메모리에 데이터가 가득차 더 이상 새로운 데이터를 추가하지 못하는 경우
- 이를 해결하기 위해 LoRA/QLoRA를 사용하여 연산과, 모델 크기를 줄여서 사용한다
### 풀 파인튜닝
- 모델의 모든 파라미터를 수정하는 방식
  - 메모리 효율 낮음
  - 성능은 좋음
- LoRA : 모델의 일부 파라미터만 수정하는 방식
  - 메모리 효율 높임
  - 성능은 풀 파인튜닝보다 약간 낮음